# 2. Data and Clear Data
In the previous notebook, "first analysis data," we examined the data as follows:  
1. Data type  
2. Data issues and problems  
3. Comparing data with each other  
4. Plotting charts from the data  
5. Assessing the data distribution to determine if it is normal or not  
6. Analyzing data behavior before movement, during movement, and after movement  
7. Identifying the type of movement and the dominant foot (which foot bears the most force and pulls the movement toward itself)  
8. Examining movement behavior  

In [54]:
#basic
import os
import math
import gpxpy # read .GPX
from pathlib import Path

#analysis
import numpy as np
import pandas as pd

#Statistik
import scipy.stats as stats

#plot
import matplotlib.pyplot as plt
import seaborn as sns

one file for sampel

In [5]:
# file_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-11 09-38-11-074_Asphalt.txt"
file_path =  r"C:/Users/user/Desktop/Fatemeh/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-17 10-09-36-341.txt"

with open(file_path, 'r', encoding='utf-8') as f:
    first_two_lines = [next(f) for _ in range(2)]

for i, line in enumerate(first_two_lines, 1):
    print(f"line {i}: {line.rstrip()}")

# name file
print(first_two_lines[0][5:].replace(".pdo\n",""))
# name Type of surface and last number
print(first_two_lines[1][8:].replace("\n","").split("_"))

line 1: File: loadsol_25-11-17 10-09-36-341.pdo
line 2: Comment:S07_Asphalt_01
 loadsol_25-11-17 10-09-36-341
['S07', 'Asphalt', '01']


# Force Data of Foot

In [32]:
# file_path =  r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-17 10-09-36-341.txt"
file_path =  r"C:/Users/user/Desktop/Fatemeh/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-17 10-09-36-341.txt"

def make_dataframe(path):
    """find data in text
    Data = read file
    path = path of file
    line1 = 1st line name of file
    line2 = 2st line name of person and surface and one number
    line3 = 3rd line name of columns
    line4 = 4th line name of columns
    """
    #read data and line 1,2 ,3 4
    with open(path, "r", encoding="utf-8") as f:
        line1 = f.readline().rstrip("\n")
        line2 = f.readline().rstrip("\n")
        line3 = f.readline().rstrip("\n")
        line4 = f.readline().rstrip("\n")
    # line 1 and 2 and make name of columns
    a1 = np.array(line3.split("\t"))
    a2 = np.array(line4.split("\t"))[0:len(a1)]
    b = a1 + a2
    # replace name of columns
    name_colunms = np.char.replace(b, "Force[N]", "")
    name_colunms = np.char.replace(name_colunms, "::", "_")
    name_colunms = np.char.replace(name_colunms, "-", "_")
    #clear name of colums
    left_L =[]
    right_R = []
    for i in range(len(name_colunms)):
        left_L.append(name_colunms[i].rfind("_L"))
        right_R.append(name_colunms[i].rfind("_R"))
    for j in range(len(name_colunms)):
        if left_L[j] > 0:
            name_colunms[j] = name_colunms[j][left_L[j]+1:]
        elif right_R[j]>0:
            name_colunms[j] = name_colunms[j][right_R[j]+1:]
    name_colunms = np.char.replace(name_colunms, "L ", "L")
    # make name of dataframe
    name = line1[-7:-4]
    person_surface_number = line2[8:].split("_")
    main_name = (person_surface_number[1] + "_Foot_" +
                 person_surface_number[0]+ "_" +
                 person_surface_number[2]+ "_" +
                 name)
    #read row 5 to end of data
    df = pd.read_csv(path,
                     skiprows=4,
                     sep=r"\s+",
                     decimal=",",
                     header=None)
    # rename columns of dataframe
    df.columns = name_colunms.tolist()
    df = df.rename(columns={"L":"Total_L_Foot", "R":"Total_R_Foot"})
    # drop and make time
    Time = np.arange(len(df)) * 0.01
    df = df.drop(columns="Time[secs]")
    df.insert(loc=0, column='Time', value=Time)
    #raplace -1 to 0
    df = df.replace(-1, 0)
    # make columns missed
    if len(df.columns)== 7:
        df.insert(loc=2, column= "L_Midfoot", value=0)
        df.insert(loc=6, column= "R_Midfoot", value=0)

    return name, person_surface_number, name_colunms, main_name , df

# q ,w,  s , z, x= make_dataframe(file_path)
# globals()[z] = x
# globals()[z]

# make_dataframe(file_path)
name ,person_surface_number, name_colunms, main_name, data_frame= make_dataframe(file_path)

main_name

'Asphalt_Foot_S07_01_341'

Read all data - Force data Foot

In [33]:
# read data form
input_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main"
# save datafrom to folder as xlsx
output_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Clear data as CSV"

os.makedirs(input_path, exist_ok=True)
file_names = [f for f in os.listdir(input_path) if f.endswith('.txt')]

# test read data
# file_names[0]
# path = f"{input_path}/{file_names[0]}"
# q ,w,  s , z, x= make_dataframe(path)

if 'name' in globals():
    del name
if 'person_surface_number' in globals():
    del person_surface_number
if 'person_surface_number' in globals():
    del name_colunms
if 'main_name' in globals():
    del main_name
if 'data_frame' in globals():
    del data_frame

name_of_dataframe = []
for file_name in file_names:
    path = f"{input_path}/{file_name}"
    print(path)
    name ,person_surface_number, name_colunms, main_name, data_frame= make_dataframe(path)
    print(main_name)
    name_of_dataframe.append(main_name)
    globals()[main_name] = data_frame
    # save data as CSV
    output_file = os.path.join(output_path, f"{main_name}.csv")
    globals()[main_name].to_csv(output_file, index=False)


C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 09-24-19-079.txt
Gravel_Foot_P02_01_079
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 09-38-11-074.txt
Asphalt_Foot_P02_01_074
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 09-52-01-474.txt
Sand_Foot_P02_01_474
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 10-06-34-305.txt
Grass_Foot_P02_01_305
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 10-09-51-519.txt
Grass_Foot_P02_02_519
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Ma

# Heartrate (Polar Ignite 2)

In [48]:
# Path Folder
folder_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Heartrate (Polar Ignite 2)/Heartrate (Polar Ignite 2)"
# file name
file_name = "Maximilian_Stasica_2025-11-16_14-48-22"
# path of file
full_file_path = f"{folder_path}/{file_name}.csv"
# read file
df = pd.read_csv(full_file_path)

In [49]:
df

,Name,Sport,Date,Start time,Duration,Total distance (km),Average heart rate (bpm),Average speed (km/h),Max speed (km/h),Average pace (min/km),...,Descent (m),Average power (W),Max power (W),Notes,Height (cm),Weight (kg),HR max,HR sit,VO2max,Unnamed: 29
0,Maximilian Stasica,RUNNING,16-11-2025,14:48:22,01:12:39,3.15,91,2.6,22.6,23:04,...,115.0,NaN,NaN,NaN,175.0,88.0,190.0,55.0,47.0,NaN
1,Sample rate,Time,HR (bpm),Speed (km/h),Pace (min/km),Cadence,Altitude (m),Stride length (m),Distances (m),Temperatures (C),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,00:00:00,80,0.1,00:00,0,NaN,NaN,0.00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,00:00:01,81,0.4,145:15,0,NaN,NaN,0.00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,00:00:02,81,0.0,00:00,0,NaN,NaN,0.00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4355,NaN,01:12:33,76,0.0,00:00,0,150,NaN,3149.80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4356,NaN,01:12:34,76,0.0,00:00,0,150,NaN,3149.80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4357,NaN,01:12:35,77,0.0,00:00,0,150,NaN,3149.80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4358,NaN,01:12:36,77,0.0,00:00,0,150,NaN,3149.80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
folder_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Heartrate (Polar Ignite 2)/Heartrate (Polar Ignite 2)"
# file name
file_name = "Maximilian_Stasica_2025-11-16_14-48-22.GPX"
# path of file
gpx_file_path = f"{folder_path}/{file_name}"

with open(gpx_file_path, 'r', encoding='utf-8') as gpx_file:
    gpx = gpxpy.parse(gpx_file)

# نمایش اطلاعات کلی
print(f"تعداد trackها: {len(gpx.tracks)}")

for track in gpx.tracks:
    print(f"Track: {track.name}")
    for segment in track.segments:
        print(f"  تعداد نقاط در این segment: {len(segment.points)}")
        for point in segment.points[:3]:  # نمایش ۳ نقطه اول به عنوان نمونه
            print(f"    lat: {point.latitude}, lon: {point.longitude}, elevation: {point.elevation}, time: {point.time}")

تعداد trackها: 1
Track: None
  تعداد نقاط در این segment: 4342
    lat: 49.895375, lon: 8.80459667, elevation: 375.0, time: 2025-11-16 13:48:37.673000+00:00
    lat: 49.89566333, lon: 8.80445, elevation: 368.0, time: 2025-11-16 13:48:41.672000+00:00
    lat: 49.895665, lon: 8.804505, elevation: 363.0, time: 2025-11-16 13:48:42.672000+00:00


In [55]:
import pandas as pd
import gpxpy
from pathlib import Path

folder_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Heartrate (Polar Ignite 2)/Heartrate (Polar Ignite 2)"
# file name
file_name = "Maximilian_Stasica_2025-11-16_14-48-22.GPX"
# path of file
gpx_path = f"{folder_path}/{file_name}"

# خواندن فایل GPX
with open(gpx_path, 'r', encoding='utf-8') as f:
    gpx = gpxpy.parse(f)

# استخراج داده‌ها
data = []
for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            data.append({
                'time': point.time,
                'latitude': point.latitude,
                'longitude': point.longitude,
                'elevation': point.elevation
            })

# تبدیل به DataFrame
df = pd.DataFrame(data)

# نمایش اطلاعات
print(df.head())

                              time   latitude  longitude  elevation
0 2025-11-16 13:48:37.673000+00:00  49.895375   8.804597      375.0
1 2025-11-16 13:48:41.672000+00:00  49.895663   8.804450      368.0
2 2025-11-16 13:48:42.672000+00:00  49.895665   8.804505      363.0
3 2025-11-16 13:48:43.673000+00:00  49.895693   8.804558      355.0
4 2025-11-16 13:48:44.673000+00:00  49.895725   8.804515      347.0
